In [43]:
mapping = {
    'а': 'ɑ', 'б': 'b', 'в': 'v', 'г': 'ɡ', 'д': 'd', 'е': 'e',
    'ж': 'ʤ', 'з': 'z', 'и': 'i', 'й': 'j', 'к': 'k', 'л': 'l',
    'м': 'm', 'н': 'n', 'ң': 'ŋ', 'о': 'o', 'ө': 'ø', 'п': 'p',
    'р': 'ɾ', 'с': 's', 'т': 't', 'у': 'u', 'ү': 'y', 'ф': 'f',
    'х': 'χ', 'ч': 'ʧ', 'ш': 'ʃ', 'щ': 'ɕ', 'ь': 'ʲ', 'ы': 'ɯ',
    'ъ': '0', 'э': 'e', 'ю': 'ju'
}

def transliterate(text):
    result = ""
    for ch in text:
        # заменяем букву, если она есть в словаре (учитываем нижний регистр)
        result += mapping.get(ch.lower(), ch)
    return result

text = "элка^рт ка^рталары боюнча^ лимитте^р , банкоматтарда^ , күндү^з ..."
converted = transliterate(text)
print(converted)


elkɑ^ɾt kɑ^ɾtɑlɑɾɯ bojunʧɑ^ limitte^ɾ , bɑnkomɑttɑɾdɑ^ , kyndy^z ...


In [4]:
import subprocess
import shlex
from pathlib import Path

from pathlib import Path
import os


TIMEOUT = 10  # секунд на один запуск (на случай зависания)
HFST_CMD = 'hfst-lookup dev/ortho/my_cyr-ipa.ohfst'
WORKDIR = Path(os.getcwd()).resolve()


def run_lookup(text: str, timeout: int = TIMEOUT) -> dict:
    """
    Запускает: echo "text" | hfst-lookup cyr-ipa.ohfst
    """
    # формируем команду как shell pipeline
    cmd = f'echo {shlex.quote(text.lower())} | {HFST_CMD}'
    try:
        proc = subprocess.run(cmd, shell=True, cwd=str(WORKDIR),
                              capture_output=True, text=True, timeout=timeout)

        return choose_var(proc.stdout.strip(), text)
    except subprocess.TimeoutExpired as e:
        return {"stdout": "", "stderr": f"TIMEOUT after {timeout}s", "rc": -1}


# функция костыльная, конечно, но что поделать, мы все не идеальны
def choose_var(output, text):

    output_stressed = [x for x in list(set(output.split('\n'))) if "ˈ" in x] #

    # если несколько вариантов с ударением просто берем первое слово в списке
    if len(output_stressed) > 1:
        print("len(output) > 1:", output_stressed,)
        output_stressed = output_stressed[0].split('\t')[1]

    # если слово без ударения или не разметилось
    elif len(output_stressed) < 1:
        target_word = output.split('\n')[0].split('\t')[1]

        # слово без ударения
        if target_word[-1] != '?':
            output_stressed = target_word

        # слово не разметилось, мапим буква к букве тогда
        else:
            print(output, target_word, text, transliterate(text))
            output_stressed = transliterate(text)

    else:
        # идеальный разбор
        output_stressed = output_stressed[0].split('\t')[1]

    if text.isupper() == True:
        output_stressed = output_stressed.upper()
    return output_stressed


import re
path_data = 'filtered_manifest.txt'

with open(path_data, 'r', encoding='utf-8') as f:
    file = f.readlines()


all_data = ''

for x in file:
    text = x.split('|')[4]
    # print(text)
    tokens = re.findall(r"[а-яА-ЯёЁөңү^]+|[.,!?;:]", text)
    phonemazed_text = []
    for tok in tokens:
        if tok not in '.,!?;:':
            phonemazed_text.append(run_lookup(tok))
        else:
            phonemazed_text.append(tok)


    all_data += x.strip('\n') + '|' + ' '.join(phonemazed_text).replace('ˈ', '') + '\n'  # апертиум сам проставляет ударения. Удаляем все его ударения ˈ , оставляем только наши ^
    #print(phonemazed_text)

In [5]:
len(file)

5

In [6]:
with open('filtered_manifest_phonemazed.txt', 'w', encoding='utf-8') as f:
    f.write(all_data)

In [38]:
all_data.split('\n')[3]

'DUMMY1/00003_003_sales_001_commas_sep-2_1_Aiganysh_neutral.wav|Aiganysh|neutral|ЭЛКАРТ карталары боюнча лимиттер: банкоматтарда: күндүз 100 000 сомго чейин – акысыз , 100 000 сомдон ашык акчаны банкоматтан алуу мүмкүн эмес.|элка^рт ка^рталары боюнча^ лимитте^р , банкоматтарда^ , күндү^з жү^з ми^ң сомго^ чейи^н акысы^з , жү^з ми^ң сомдо^н ашы^к акчаны^ банкоматта^н алуу^ мүмкү^н эме^с.|elqɑ^ɾt qɑ^ɾtɑɫɑˈɾɯ bojunʧɑ^ limitte^ɾ , bɑnqomɑttɑɾdɑ^ , kyndy^z ʤy^z mi^ŋ somɢo^ ʧeji^n ɑqɯsɯ^z , ʤy^z mi^ŋ somdo^n ɑʃɯ^q ɑqʧɑnɯ^ bɑnqomɑttɑ^n ɑɫuː^ mymky^n eme^s .'